# EXP-002A — EXP-001 no-undo ablation

Small controlled ablation. Keep EXP-001 random visible-pixel policy, but remove ACTION7 / undo from the random action pool.

Validation gate: local score >= 0.2124 and levels >= 7/183 before submission.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:
import os, json, random, zlib
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, dotenv, arc_agi
from arcengine import GameAction, GameState

EXP_ID='EXP-002A'
MAX_MOVES=1000
SEED=42
USE_PER_GAME_SEED=False
WORK=Path('/kaggle/working')
ART=WORK/'exp002a_artifacts'
ART.mkdir(exist_ok=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    get_ipython().system('curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games')
    mode='online'; envdir=''
else:
    mode='offline'; envdir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/'

(WORK/'.env').write_text(f'''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE={mode}\nENVIRONMENTS_DIR={envdir}\nRECORDINGS_DIR=/kaggle/working/server_recording\n''')
dotenv.load_dotenv(WORK/'.env', override=True)

def rng_for(g):
    return random.Random(SEED if not USE_PER_GAME_SEED else SEED+(zlib.adler32(g.encode())&0xffffffff))

def pix(frame,rng):
    color=rng.choice(np.unique(frame).tolist())
    ys,xs=np.where(frame==color)
    i=rng.randint(0,len(xs)-1)
    return {'x':int(xs[i]),'y':int(ys[i])}

def play(env,g):
    rng=rng_for(g)
    r=env._last_response
    last=None
    action_counts=defaultdict(int)
    # EXP-002A ablation: keep EXP-001 random policy but exclude RESET and ACTION7/undo.
    acts=[a for a in GameAction if a not in (GameAction.RESET, GameAction.ACTION7)]
    for m in range(1,MAX_MOVES+1):
        if r is None or r.state==GameState.WIN:
            break
        if r.state in (GameState.GAME_OVER,GameState.NOT_PLAYED):
            r=env.step(GameAction.RESET,{})
            last='RESET'
            action_counts['RESET']+=1
            continue
        a=rng.choice(acts)
        data=pix(r.frame[-1],rng) if a.is_complex() and r.frame else {}
        r=env.step(a,data,reasoning={'exp_id':EXP_ID,'policy':'exp001_random_visible_pixel_no_undo'})
        last=a.name
        action_counts[a.name]+=1
    return {'game_id':g,'moves':m,'state':r.state.name if r else 'unknown','levels_completed':r.levels_completed if r else 0,'last_action':last,'action_counts':dict(action_counts)}

arcade=arc_agi.Arcade()
infos=list(arcade.available_environments)
rows=[]
details=[]
print(EXP_ID,'envs=',len(infos),'MAX_MOVES=',MAX_MOVES,'SEED=',SEED,'USE_PER_GAME_SEED=',USE_PER_GAME_SEED)
for i,info in enumerate(infos,1):
    row=play(arcade.make(info.game_id), info.game_id)
    details.append(row)
    flat={k:v for k,v in row.items() if k!='action_counts'}
    rows.append(flat)
    print(f'[{i:03d}/{len(infos):03d}] {flat}')

pd.DataFrame(rows).to_csv(ART/'exp002a_run_results.csv', index=False)
(ART/'exp002a_run_details.json').write_text(json.dumps(details,indent=2))

action_totals=defaultdict(int)
for d in details:
    for k,v in d['action_counts'].items(): action_totals[k]+=int(v)
(ART/'exp002a_action_counts.json').write_text(json.dumps(dict(action_totals),indent=2,sort_keys=True))

summary={'exp_id':EXP_ID,'max_moves':MAX_MOVES,'seed':SEED,'use_per_game_seed':USE_PER_GAME_SEED,'change':'EXP-001 visible-pixel random baseline excluding ACTION7/undo'}
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sc=arcade.get_scorecard()
    summary.update(score=float(sc.score),total_environments_completed=int(sc.total_environments_completed),total_environments=int(sc.total_environments),total_levels_completed=int(sc.total_levels_completed),total_levels=int(sc.total_levels),total_actions=int(sc.total_actions))
    pd.DataFrame([{'game':e.id,'score':float(e.score),'levels_completed':int(e.levels_completed),'actions':int(e.actions),'completed':bool(e.completed)} for e in sc.environments]).to_csv(ART/'exp002a_scorecard_by_environment.csv',index=False)
    pd.DataFrame([{'tag':t.id,'score':float(t.score),'levels_completed':int(t.levels_completed),'number_of_environments':int(t.number_of_environments)} for t in (sc.tags_scores or [])]).to_csv(ART/'exp002a_scorecard_by_tag.csv',index=False)
    print('Score:', f'{sc.score:.4f}', 'Levels:', f'{sc.total_levels_completed}/{sc.total_levels}', 'Actions:', sc.total_actions)

(ART/'exp002a_scorecard_summary.json').write_text(json.dumps(summary,indent=2))
sp=WORK/'submission.parquet'
if not sp.exists():
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(sp,index=False)
manifest=sorted(str(p) for p in ART.glob('*'))
pd.DataFrame({'artifact':manifest}).to_csv(ART/'artifact_manifest.csv',index=False)
print('submission:', sp, sp.exists())
print('artifacts:', ART)
print(json.dumps(summary,indent=2))
summary
